# Image Classification with EfficientNet-B0 Fine-Tuning

## Overview

This notebook demonstrates how to fine-tune a pretrained **EfficientNet-B0** model on the
[Microsoft Cats vs Dogs](https://www.kaggle.com/datasets/shaunthesheep/microsoft-catsvsdogs-dataset)
dataset for binary image classification.

**Pipeline:**
1. Load and preprocess the Cats-vs-Dogs dataset with train/validation splits.
2. Build an EfficientNet-B0 model (pretrained on ImageNet) via the `timm` library.
3. Freeze the backbone and train only the classifier head.
4. Unfreeze the last few layers and fine-tune end-to-end with a lower learning rate.
5. Evaluate with a confusion matrix and classification report.

**Dataset:** ~25,000 labeled images of cats and dogs.  
**Model:** EfficientNet-B0 (5.3M parameters) from `timm`.  
**Framework:** PyTorch 2.x

In [ ]:
# ============================================================
# Cell 2: Install & Import
# ============================================================

!pip install -q timm

import os
import copy
import random
import numpy as np
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

import torchvision
from torchvision import transforms, datasets

import timm
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)
from tqdm.auto import tqdm

print(f"PyTorch version : {torch.__version__}")
print(f"torchvision     : {torchvision.__version__}")
print(f"timm            : {timm.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

In [ ]:
# ============================================================
# Cell 3: Configuration
# ============================================================

# -- Hyperparameters --
IMG_SIZE     = 224
BATCH_SIZE   = 32
NUM_EPOCHS   = 5        # Phase 1: frozen backbone
FT_EPOCHS    = 3        # Phase 2: fine-tune unfrozen layers
LR           = 1e-3     # Phase 1 learning rate
FT_LR        = 1e-4     # Phase 2 learning rate
NUM_CLASSES  = 2
NUM_WORKERS  = 2
SEED         = 42

# -- Paths --
DATA_DIR = Path("/kaggle/input/microsoft-catsvsdogs-dataset/PetImages")

# -- Device --
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# -- Reproducibility --
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

In [ ]:
# ============================================================
# Cell 4: Data Loading
# ============================================================

# ImageNet statistics for normalization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Training transforms: augmentation + normalization
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Validation transforms: deterministic resize + normalization only
val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def is_valid_image(path: str) -> bool:
    """Filter out corrupt / non-image files that ship with this dataset."""
    try:
        img = Image.open(path)
        img.verify()
        return True
    except Exception:
        return False


# Load full dataset (both Cat and Dog subfolders)
full_dataset = datasets.ImageFolder(root=str(DATA_DIR), is_valid_file=is_valid_image)

class_names = full_dataset.classes  # ['Cat', 'Dog']
print(f"Classes: {class_names}")
print(f"Total valid images: {len(full_dataset)}")

# ---------- Train / Val split (80/20, stratified) ----------
from sklearn.model_selection import train_test_split

targets = [s[1] for s in full_dataset.samples]
indices = list(range(len(full_dataset)))

train_idx, val_idx = train_test_split(
    indices, test_size=0.2, stratify=targets, random_state=SEED
)

print(f"Train size: {len(train_idx)} | Val size: {len(val_idx)}")


class TransformSubset(torch.utils.data.Dataset):
    """Wraps a Subset so we can apply different transforms to train/val."""
    def __init__(self, dataset, indices, transform=None):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        path, label = self.dataset.samples[self.indices[idx]]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label


train_dataset = TransformSubset(full_dataset, train_idx, transform=train_transforms)
val_dataset   = TransformSubset(full_dataset, val_idx,   transform=val_transforms)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

In [ ]:
# ============================================================
# Cell 5: Visualize Sample Images
# ============================================================

def denormalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    """Reverse ImageNet normalization for display."""
    mean = torch.tensor(mean).view(3, 1, 1)
    std  = torch.tensor(std).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)


# Grab one batch from the training loader
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    img = denormalize(images[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(class_names[labels[i].item()], fontsize=14)
    ax.axis("off")

fig.suptitle("Sample Training Images", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 6: Model Setup - EfficientNet-B0
# ============================================================

def build_model(num_classes: int, freeze_backbone: bool = True):
    """
    Load a pretrained EfficientNet-B0 from timm and replace
    the classifier head for our binary task.
    """
    model = timm.create_model("efficientnet_b0", pretrained=True)

    # Freeze all backbone parameters
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # Replace the classifier head
    in_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes),
    )

    # Classifier head is always trainable
    for param in model.classifier.parameters():
        param.requires_grad = True

    return model


model = build_model(NUM_CLASSES, freeze_backbone=True).to(DEVICE)

# Quick summary
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")
print(f"Frozen parameters    : {total_params - trainable_params:,}")

In [ ]:
# ============================================================
# Cell 7: Training Loop (Phase 1 - Frozen Backbone)
# ============================================================

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)


def train_one_epoch(model, loader, criterion, optimizer, device):
    """Train for a single epoch. Returns average loss and accuracy."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="  Train", leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc  = correct / total
    return epoch_loss, epoch_acc


@torch.no_grad()
def validate(model, loader, criterion, device):
    """Validate. Returns average loss, accuracy, all predictions and labels."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds  = []
    all_labels = []

    for images, labels in tqdm(loader, desc="  Val  ", leave=False):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / total
    epoch_acc  = correct / total
    return epoch_loss, epoch_acc, np.array(all_preds), np.array(all_labels)


# ---------- Phase 1 Training ----------
best_val_acc = 0.0
best_model_state = None
history = defaultdict(list)

print("=" * 60)
print("Phase 1: Training classifier head (backbone frozen)")
print("=" * 60)

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, DEVICE
    )
    val_loss, val_acc, _, _ = validate(
        model, val_loader, criterion, DEVICE
    )

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"  Train Loss: {train_loss:.4f}  |  Train Acc: {train_acc:.4f}")
    print(f"  Val   Loss: {val_loss:.4f}  |  Val   Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = copy.deepcopy(model.state_dict())
        print(f"  --> New best model (val acc: {best_val_acc:.4f})")

In [ ]:
# ============================================================
# Cell 8: Unfreeze & Fine-Tune (Phase 2)
# ============================================================

# Restore best Phase 1 weights before fine-tuning
model.load_state_dict(best_model_state)

# Unfreeze the last two blocks of the backbone
# EfficientNet-B0 blocks are in model.blocks (list of 7 block groups)
for param in model.blocks[-2:].parameters():
    param.requires_grad = True

# Also unfreeze batch-norm in the conv_head
for param in model.conv_head.parameters():
    param.requires_grad = True
for param in model.bn2.parameters():
    param.requires_grad = True

trainable_ft = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters after unfreeze: {trainable_ft:,}")

# Lower learning rate for fine-tuning
optimizer_ft = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=FT_LR,
)

print("\n" + "=" * 60)
print("Phase 2: Fine-tuning unfrozen layers")
print("=" * 60)

for epoch in range(1, FT_EPOCHS + 1):
    print(f"\nFine-Tune Epoch {epoch}/{FT_EPOCHS}")

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer_ft, DEVICE
    )
    val_loss, val_acc, _, _ = validate(
        model, val_loader, criterion, DEVICE
    )

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"  Train Loss: {train_loss:.4f}  |  Train Acc: {train_acc:.4f}")
    print(f"  Val   Loss: {val_loss:.4f}  |  Val   Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = copy.deepcopy(model.state_dict())
        print(f"  --> New best model (val acc: {best_val_acc:.4f})")

In [ ]:
# ============================================================
# Cell 9: Evaluation
# ============================================================

# Load best weights for final evaluation
model.load_state_dict(best_model_state)

val_loss, val_acc, all_preds, all_labels = validate(
    model, val_loader, criterion, DEVICE
)
print(f"Final Validation Accuracy: {val_acc:.4f}\n")

# ---------- Classification Report ----------
print(classification_report(all_labels, all_preds, target_names=class_names))

# ---------- Confusion Matrix ----------
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Confusion Matrix", fontsize=14)
plt.tight_layout()
plt.show()

# ---------- Plot Training History ----------
epochs_range = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_range, history["train_loss"], "o-", label="Train Loss")
ax1.plot(epochs_range, history["val_loss"],   "o-", label="Val Loss")
ax1.axvline(x=NUM_EPOCHS + 0.5, color="gray", linestyle="--", alpha=0.5,
            label="Unfreeze")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss Curve")
ax1.legend()

ax2.plot(epochs_range, history["train_acc"], "o-", label="Train Acc")
ax2.plot(epochs_range, history["val_acc"],   "o-", label="Val Acc")
ax2.axvline(x=NUM_EPOCHS + 0.5, color="gray", linestyle="--", alpha=0.5,
            label="Unfreeze")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy Curve")
ax2.legend()

plt.suptitle("Training History (Phase 1 | Phase 2)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# ---------- Prediction Samples with Confidence ----------
model.eval()
images_batch, labels_batch = next(iter(val_loader))
images_batch = images_batch.to(DEVICE)

with torch.no_grad():
    logits = model(images_batch)
    probs  = torch.softmax(logits, dim=1)
    confs, preds = torch.max(probs, dim=1)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    img = denormalize(images_batch[i].cpu()).permute(1, 2, 0).numpy()
    pred_label = class_names[preds[i].item()]
    true_label = class_names[labels_batch[i].item()]
    confidence = confs[i].item() * 100

    color = "green" if pred_label == true_label else "red"
    ax.imshow(img)
    ax.set_title(
        f"Pred: {pred_label} ({confidence:.1f}%)\nTrue: {true_label}",
        fontsize=11, color=color,
    )
    ax.axis("off")

fig.suptitle("Predictions with Confidence Scores", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 10: Save Best Model Weights
# ============================================================

SAVE_DIR = Path("/kaggle/working")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

save_path = SAVE_DIR / "efficientnet_b0_catsvsdogs_best.pth"

torch.save(
    {
        "model_state_dict": best_model_state,
        "class_names": class_names,
        "img_size": IMG_SIZE,
        "num_classes": NUM_CLASSES,
        "best_val_acc": best_val_acc,
    },
    save_path,
)

file_size_mb = save_path.stat().st_size / (1024 * 1024)
print(f"Model saved to: {save_path}")
print(f"File size: {file_size_mb:.1f} MB")
print(f"Best validation accuracy: {best_val_acc:.4f}")